# IT ticket triage: QLoRA fine-tuning on a Colab T4

Runs the whole pipeline end to end: **install → log in → data → baselines → QLoRA training → evaluation → publish**.

Before you start:
1. **Runtime → Change runtime type → T4 GPU.**
2. Add a Hugging Face **write** token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar,
   enable "Notebook access"). Create one at https://huggingface.co/settings/tokens.
3. `REPO_URL` below already points to https://github.com/ShreyaBatra6/it-ticket-triage-lora (change it only for a fork).

Timings measured on a T4: install 3 min · data 1 min · baselines 15-20 min · training 45 min ·
evaluation 5 min · merge + publish 20 min (about 2 hours in total).

Run the cells in order and keep this tab open: a free Colab session can disconnect, and a new session
starts with no files, so the setup cells (1-3) and the data cell must be run again.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU (or the free GPU quota is used up)."
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

No GitHub repo yet? Zip the project folder on your machine (without `.venv`, `data`, `outputs`), upload it
with the Files sidebar, and run the commented `unzip` line instead of `git clone`.

In [ ]:
import os

REPO_URL = "https://github.com/ShreyaBatra6/it-ticket-triage-lora"  # change only if you use a fork
WORKDIR = "/content/it-ticket-triage-lora"

if not os.path.exists(WORKDIR):
    !git clone -q {REPO_URL} {WORKDIR}
    # !unzip -q /content/it-ticket-triage-lora.zip -d /content/
%cd {WORKDIR}

import sys

if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)  # lets later cells `import config` etc. (%cd alone does not)

## 2. Install

Colab already has a CUDA build of PyTorch; `requirements.txt` pins everything else.
pip may print warnings about other preinstalled Colab packages; they do not affect this project.

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch, transformers, peft, trl, bitsandbytes as bnb; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__, '| bitsandbytes', bnb.__version__)"

## 3. Log in to Hugging Face

In [ ]:
import os

from google.colab import userdata
from huggingface_hub import login, whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # child processes (make, python) inherit it
login(token=os.environ["HF_TOKEN"])
print("Logged in as", whoami()["name"])

## 4. (Optional) Smoke test

The same tiny-model pipeline that runs on CPU locally, about 5 minutes here. Skip it if you are in a hurry.

In [ ]:
# !make smoke

## 5. Data

Generates ~4,000 synthetic tickets, dedupes, splits (template held out) and pushes the dataset to the Hub.

In [ ]:
!make data-push
!python -c "import json; s = json.load(open('data/processed/stats.json')); print('split sizes:', s['split_sizes']); print('split method:', s['stratified_by'])"

## 6. Baselines: before fine-tuning

First a 20-ticket check that the base model loads and generates, then the three "before" rows of the
results table: TF-IDF + LogReg, base model zero-shot, base model few-shot (k=3).

In [ ]:
!python eval.py --mode zero-shot --limit 20 --run-name _check_base
!rm -f results/_check_base*

In [ ]:
!make baseline

## 7. Fine-tune with QLoRA

A 20-step run first checks the 4-bit + fp16 path on this GPU in about 5 minutes; then the full run
(200 steps, about 1 epoch, 45 minutes). Watch the curves live in TensorBoard.

In [ ]:
# Quick check only: this cell deletes its adapter afterwards. The real run is `!make train` below.
!python train_lora.py --max-steps 20
!rm -rf outputs/checkpoints outputs/lora-adapter

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/tensorboard

In [ ]:
!make train

## 8. Evaluate the fine-tuned model

In [ ]:
!make eval

In [ ]:
import os
import sys

os.chdir("/content/it-ticket-triage-lora")  # also works after a session restart
sys.path.insert(0, os.getcwd())

import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from config import load_config

cfg = load_config()
rows = []
for spec in cfg.eval.report_rows:
    path = Path("results") / f"{spec.run_name}.json"
    if path.exists():
        m = json.loads(path.read_text())["metrics"]
        rows.append({"Model": spec.label, "Cat Acc": m["category_accuracy"], "Cat F1": m["category_macro_f1"],
                     "Prio Acc": m["priority_accuracy"], "Prio F1": m["priority_macro_f1"],
                     "Exact Match": m["exact_match"], "Valid JSON %": m["valid_json_rate"]})
display(pd.DataFrame(rows).set_index("Model").style.format("{:.3f}"))
for name in ("lora_finetuned_cm_category.png", "lora_finetuned_cm_priority.png", "train_loss.png"):
    display(Image(filename=f"results/{name}", width=520))

## 9. Publish

1. **Dry run:** builds both model cards and the merged model locally in `outputs/hub/`. Read the card below.
2. **Optional:** evaluate the merged bf16 weights directly, so the merged model's card reports its own scores.
3. **Publish:** pushes the adapter and the merged model. (`!make publish-space` also deploys the Gradio
   demo as a Space, which needs a Hugging Face PRO account.)

In [ ]:
!make publish-dry
from pathlib import Path

from IPython.display import Markdown

Markdown(Path("outputs/hub/adapter/README.md").read_text().split("---", 2)[2][:6000])

In [ ]:
!make eval-merged

In [ ]:
!make publish

## 10. Bring the results back to GitHub

Refreshes the results table in `README.md`, then downloads `results/` + `README.md` as a zip.
Unzip it into your local repo and commit (or commit from here if you cloned with push access).

In [ ]:
!make readme
!zip -qr results_bundle.zip results README.md
from google.colab import files
files.download("results_bundle.zip")

## 11. (Optional) Try the demo

Loads the merged model you just published from the Hub and opens the Gradio app below, plus a
temporary public `*.gradio.live` link. Good moment for a screenshot.

In [ ]:
import os
import sys

os.chdir("/content/it-ticket-triage-lora")
sys.path.insert(0, os.getcwd())

import app
from config import load_config

cfg = load_config()
triage = app.Triage(cfg, cfg.hub.merged_repo, None)
app.build_ui(triage, cfg).launch(share=True)